In [ ]:
import polars as pl

data = pl.read_parquet("../data/1d/1d.parquet").lazy()

In [ ]:
data.collect_schema()

In [ ]:
def compute_daily_zscore_factors(
    data: pl.LazyFrame,
    reference_symbol: str = "BTCUSDT",
    start_date=None,
    end_date=None,
    window_size: int = 10,
):
    """
    Compute daily z-scores for all symbols against a reference symbol.

    Parameters:
    -----------
    data : polars.LazyFrame
        Price data with columns: symbol, open_time, close
    reference_symbol : str
        Reference symbol (e.g., 'BTCUSDT') - will be filled with zeros
    start_date : datetime
        Start date for calculation
    end_date : datetime
        End date for calculation
    window_size : int
        Rolling window size in days (default: 20)

    Returns:
    --------
    polars.DataFrame with columns: date, symbol, zscore
    """

    # Filter date range if provided
    filtered_data = data
    if start_date is not None:
        filtered_data = filtered_data.filter(pl.col("open_time") >= start_date)
    if end_date is not None:
        filtered_data = filtered_data.filter(pl.col("open_time") < end_date)

    # Prepare daily data for all symbols
    daily_data = filtered_data.with_columns(
        pl.col("close").last().log().alias("log_price"),
        pl.col("close").last().alias("close_price"),
    )

    # Get reference symbol data
    ref_data = daily_data.filter(pl.col("symbol") == reference_symbol).select(
        pl.col("open_time"),
        pl.col("log_price").alias("S_ref"),
    )

    # Join all symbols with reference
    combined = daily_data.join(ref_data, on="open_time",maintain_order="left").with_columns(
        pl.col("log_price").alias("S_symbol")
    )

    # Compute z-scores using window functions over partitions
    result = (
        combined.with_columns(
            pl.rolling_cov("S_symbol", "S_ref", window_size=window_size)
            .over("symbol")
            .alias("rolling_cov"),
            pl.col("S_ref")
            .rolling_var(window_size=window_size)
            .over("symbol")
            .alias("rolling_var_ref"),
        )
        .with_columns(
            (pl.col("rolling_cov") / pl.col("rolling_var_ref")).alias("rolling_beta"),
        )
        .with_columns(
            (
                pl.col("S_symbol").rolling_mean(window_size=window_size).over("symbol")
                - pl.col("rolling_beta")
                * pl.col("S_ref").rolling_mean(window_size=window_size).over("symbol")
            ).alias("rolling_alpha"),
        )
        .with_columns(
            (
                pl.col("S_symbol")
                - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("S_ref"))
            ).alias("rolling_residual"),
        )
        .with_columns(
            (
                (
                    pl.col("rolling_residual")
                    - pl.col("rolling_residual")
                    .rolling_mean(window_size=window_size)
                    .over("symbol")
                )
                / pl.col("rolling_residual")
                .rolling_std(window_size=window_size)
                .over("symbol")
            ).alias("zscore"),
        )
        .with_columns(
            pl.when(pl.col("symbol") == reference_symbol)
            .then(0.0)
            .otherwise(pl.col("zscore"))
            .alias("zscore")
        )
        # Calculate forward returns
        .with_columns(
            pl.col("close_price").shift(-1).over("symbol").alias("close_price_next")
        )
        .with_columns(
            (pl.col("close_price_next") / pl.col("close_price") - 1).alias(
                "forward_return"
            )
        )
        # Rank IC calculation
        .with_columns(
            pl.col("zscore").rank().over("open_time").alias("alpha_rank"),
            pl.col("forward_return").rank().over("open_time").alias("return_rank"),
        )
        .collect()
    )

    # Calculate Rank IC per date
    ic_per_date = (
        result.filter(pl.col("forward_return").is_not_null())
        .group_by("open_time")
        .agg(pl.corr("alpha_rank", "return_rank").alias("rank_ic"))
        .sort("open_time")
    )

    # Calculate Rank ICIR
    mean_ic = ic_per_date["rank_ic"].drop_nans().mean()
    std_ic = ic_per_date["rank_ic"].drop_nans().std()
    rank_icir = mean_ic / std_ic if std_ic and std_ic > 0 else float("nan")

    result_final = result.select(["open_time", "symbol", "zscore"]).sort(
        ["open_time", "symbol"]
    )

    return {
        "factors": result_final,
        "ic_series": ic_per_date,
        "rank_icir": float(rank_icir),
        "mean_ic": float(mean_ic),
        "std_ic": float(std_ic),
    }


In [ ]:
compute_daily_zscore_factors(data)["rank_icir"]

In [ ]:
import zh

data_ = data.collect().join(
    compute_daily_zscore_factors(data)["factors"].with_columns(pl.col("open_time").alias("open_time")), on=["open_time", "symbol"]
)
returns = zh.backtest(data=data_)

In [ ]:
returns = zh.backtest(data=data_,plot=True,factor="zscore",mode="both",k=1)

In [ ]:
data_ =data_.filter(pl.col("open_time") >= pl.datetime(2022,1,1))

In [ ]:
returns = zh.backtest(data=data_,plot=True,reverse=True,factor="zscore",mode="both",k=5)

In [ ]:
zh.metrics(returns).collect()